# Prueba del endpoint `/llm/data-extraction`

Smoke test end-to-end del nuevo endpoint (ver `aymurai/api/endpoints/routers/llm/data_extraction/`):
llama al servidor real vía HTTP, igual que hace el frontend — `/misc/document-extract`
para obtener el texto, y `/llm/data-extraction` para la extracción + cruce con el organigrama.


In [ ]:
import json
import mimetypes
import os
import time
from pathlib import Path
from typing import Iterable

import pandas as pd
import requests
from tqdm import tqdm

API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
DOCUMENT_EXTRACT_ENDPOINT = "https://aymurai.collectiveai.io/api/misc/document-extract"
DATA_EXTRACTION_ENDPOINT = f"{API_BASE_URL}/llm/data-extraction"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/defensoria/pdfs")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "120"))

print(f"Document extract endpoint: {DOCUMENT_EXTRACT_ENDPOINT}")
print(f"Data extraction endpoint:  {DATA_EXTRACTION_ENDPOINT}")
print(f"Documentos de prueba en:   {DATA_ROOT}")


## Descubrir documentos de prueba

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    """Recursively find the documents to test under `root`.

    Args:
        root (Path): Root folder to search (recursively).
        extensions (Iterable[str]): Extensions to include (with or without leading dot).

    Returns:
        list[Path]: Matching paths, sorted alphabetically.
    """
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")
documents[:5]

## Helpers para llamar a la API

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    """
    POST a file to /misc/document-extract and return a status-wrapped payload.

    Args:
        session (requests.Session): HTTP session reused across calls.
        file_path (Path): Local path of the file to upload.

    Returns:
        dict[str, object]: {"path", "status" ("success"/"failure"), "status_code",
        "elapsed_s", "detail"} -- if status="success", detail holds
        {"document_id", "document"}; otherwise it holds the error detail.
    """
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {"file": (file_path.name, file_path.open("rb"), mime_type)}

    try:
        start = time.perf_counter()
        response = session.post(DOCUMENT_EXTRACT_ENDPOINT, files=files, timeout=REQUEST_TIMEOUT)
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload


def call_data_extraction(session: requests.Session, document: dict, **overrides) -> dict:
    """POST a Document payload to /llm/data-extraction and return the result.

    Args:
        session (requests.Session): HTTP session reused across calls.
        document (dict): Document payload ({"document", "document_id"}), as
            returned by call_extraction_api.
        **overrides: Optional request fields (model, search_backend,
            hybrid_weight, search_fields, top_k, max_retries, options).

    Returns:
        dict: The DataExtractionResult returned by the endpoint, as a dict.
    """
    payload = {"document": document, **overrides}
    response = session.post(
        DATA_EXTRACTION_ENDPOINT, json=payload, timeout=REQUEST_TIMEOUT
    )
    if not response.ok:
        print(f"Error {response.status_code}: {response.text[:500]}")
    response.raise_for_status()
    return response.json()


def summarize_destinatarios(result: dict, backend: str, top_n: int = 1) -> pd.DataFrame:
    """Flatten a data-extraction result into one row per (destinatario, field, rank).

    `candidatos_nombre` and `candidatos_cargo` are independent ranked lists --
    each produces its own rows here, tagged by `field` ("nombre" or "cargo"),
    so comparisons can filter to one field or look at both side by side.
    `top_n` controls how many candidates per field list are included (1 =
    just the best).

    Args:
        result (dict): DataExtractionResult (as a dict) returned by
            call_data_extraction.
        backend (str): Free-form label to identify this run in comparisons
            (not necessarily the actual backend name).
        top_n (int): How many candidates per field list to include (1 = best only).

    Returns:
        pd.DataFrame: One row per (destinatario, field, rank), with the
        extracted nombre/cargo, that rank's candidate, and its score. A field
        with no candidates produces a single row with rank=None.
    """
    rows = []
    for idx, dest in enumerate(result.get("destinatarios", [])):
        base = {
            "backend": backend,
            "destinatario_idx": idx,
            "nombre_extraido": dest["nombre"],
            "cargo_extraido": dest["cargo"],
            "sector": dest["sector"],
        }

        for field, key in (("nombre", "candidatos_nombre"), ("cargo", "candidatos_cargo")):
            candidatos = dest.get(key) or []
            field_base = {**base, "field": field, "n_candidatos": len(candidatos)}

            if not candidatos:
                rows.append({
                    **field_base,
                    "rank": None,
                    "candidato_nombre": None,
                    "candidato_cargo": None,
                    "score": None,
                })
                continue

            for rank, candidato in enumerate(candidatos[:top_n], start=1):
                rows.append({
                    **field_base,
                    "rank": rank,
                    "candidato_nombre": candidato["nombre"],
                    "candidato_cargo": candidato["cargo"],
                    "score": candidato["score"],
                })
    return pd.DataFrame(rows)


## Prueba sobre un solo documento

In [ ]:
session = requests.Session()

doc_path = documents[1]
print(f"Documento: {doc_path.name}")

extracted = call_extraction_api(session, doc_path)
if extracted["status"] != "success":
    raise RuntimeError(f"Extraction failed for {doc_path.name}: {extracted['detail']}")

document = extracted["detail"]
print(f"document_id: {document['document_id']}")
print(f"Párrafos extraídos: {len(document['document'])}")


In [ ]:
result_default = call_data_extraction(session, document)
print(json.dumps(result_default, indent=2, ensure_ascii=False))

## Comparar backends: fuzzy vs embeddings

In [ ]:
result_fuzzy = call_data_extraction(session, document, search_backend="fuzzy", top_k=5)
result_embeddings = call_data_extraction(session, document, search_backend="embeddings", top_k=5)

comparison = pd.concat(
    [
        summarize_destinatarios(result_fuzzy, "fuzzy", 5),
        summarize_destinatarios(result_embeddings, "embeddings", 5),
    ],
    ignore_index=True,
)
comparison

## Comparar `search_fields`: hybrid solo por cargo vs. hybrid combinada (cargo+nombre)

El endpoint ahora permite elegir qué campo(s) del destinatario se cruzan contra el
organigrama vía `search_fields` (`"nombre"`, `"cargo"`, o `"both"`). Como el nombre es
frágil frente a un cambio de gobierno (ver el análisis de la notebook `01`), esta
comparación aísla `search_fields="cargo"` contra el comportamiento combinado por
defecto (`"both"`), sobre el mismo documento y usando siempre backend `hybrid`.

In [ ]:
result_hybrid_cargo_only = call_data_extraction(
    session, document, search_backend="hybrid", search_fields="cargo", top_k=10
)
result_hybrid_both = call_data_extraction(
    session, document, search_backend="hybrid", search_fields="both", top_k=10
)

comparison_search_fields = pd.concat(
    [
        summarize_destinatarios(result_hybrid_cargo_only, "hybrid_cargo_only"),
        summarize_destinatarios(result_hybrid_both, "hybrid_both"),
    ],
    ignore_index=True,
)
comparison_search_fields


## Barrido sobre varios documentos (fuzzy vs embeddings)

In [ ]:
N_DOCS_TO_TEST = 10

all_rows = []
errors = []

for doc_path in tqdm(documents[:N_DOCS_TO_TEST], desc="Probando documentos"):
    try:
        extracted = call_extraction_api(session, doc_path)
        if extracted["status"] != "success":
            tqdm.write(f"Error de extracción en {doc_path.name}: {extracted['detail']}")
            errors.append({"doc_path": doc_path.name, "error": extracted["detail"]})
            continue

        document = extracted["detail"]
        if not document["document"]:
            tqdm.write(f"Skip (vacío): {doc_path.name}")
            continue

        result_fuzzy = call_data_extraction(session, document, search_backend="fuzzy", top_k=5)
        result_embeddings = call_data_extraction(
            session, document, search_backend="embeddings", top_k=5
        )

        df_fuzzy = summarize_destinatarios(result_fuzzy, "fuzzy")
        df_embeddings = summarize_destinatarios(result_embeddings, "embeddings")
        df_fuzzy["doc_path"] = doc_path.name
        df_embeddings["doc_path"] = doc_path.name

        all_rows.append(df_fuzzy)
        all_rows.append(df_embeddings)
    except Exception as exc:
        tqdm.write(f"Error en {doc_path.name}: {exc}")
        errors.append({"doc_path": doc_path.name, "error": str(exc)})

sweep_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
print(f"Documentos con error: {len(errors)}")
for e in errors:
    print(" ", e)
sweep_df


## Tasa de acuerdo entre backends (mismo top-1)

In [ ]:
if not sweep_df.empty:
    for field in ("nombre", "cargo"):
        value_col = "candidato_nombre" if field == "nombre" else "candidato_cargo"
        field_df = sweep_df[sweep_df["field"] == field]
        pivot = field_df.pivot_table(
            index=["doc_path", "destinatario_idx"],
            columns="backend",
            values=[value_col, "score"],
            aggfunc="first",
        )
        pivot["acuerdo_top1"] = pivot[(value_col, "fuzzy")] == pivot[(value_col, "embeddings")]
        acuerdo_top1 = pivot["acuerdo_top1"].mean()
        print(f"Acuerdo top-1 ({field}) entre fuzzy y embeddings: {acuerdo_top1:.1%}")
        display(pivot)
else:
    print("No hay datos suficientes del barrido para calcular el acuerdo.")
